In [ ]:
from platform import python_version

print(python_version())

In [4]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
    print("Running in Colab, drive mounted.")
except ModuleNotFoundError:
    print("Running locally, skipping drive.mount.")

Running locally, skipping drive.mount.


In [1]:
import os
import cv2
import numpy as np

path = r"D:\BioAnnotate_AI\Data\TaxonoML\Skin\Dataset\Train"
print("Path exists:", os.path.exists(path))

categories = os.listdir(path)
print("Found categories:", categories)

labels = list(range(len(categories)))
categoryDict = dict(zip(categories, labels))

print("Labels:", labels)
print("Category Dictionary:", categoryDict)
print("Done.")

Path exists: True
Found categories: ['1', '2', '3', '4', '5', '6', '7', '8', 'Not Applicable', 'Unknown']
Labels: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
Category Dictionary: {'1': 0, '2': 1, '3': 2, '4': 3, '5': 4, '6': 5, '7': 6, '8': 7, 'Not Applicable': 8, 'Unknown': 9}
Done.


In [3]:
import os
import cv2
import numpy as np
from tqdm import tqdm  

train_path = r"D:\BioAnnotate_AI\Data\TaxonoML\Skin\Dataset\Train"
test_path = r"D:\BioAnnotate_AI\Data\TaxonoML\Skin\Dataset\Test"

categories = os.listdir(train_path)
category_dict = dict(zip(categories, range(len(categories))))

IMG_SIZE = 128

def load_images(path, categories):
    data = []
    labels = []
    for category in categories:
        category_path = os.path.join(path, category)
        if not os.path.exists(category_path):
            print(f"Warning: {category_path} does not exist, skipping.")
            continue
        for img_name in tqdm(os.listdir(category_path), desc=f"Loading {category}"):
            img_path = os.path.join(category_path, img_name)
            try:
                img = cv2.imread(img_path)
                img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
                img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
                data.append(img)
                labels.append(category_dict[category])
            except Exception as e:
                print(f"Error loading {img_path}: {e}")
    return np.array(data), np.array(labels)

print("Loading Train dataset...")
X_train, y_train = load_images(train_path, categories)

print("Loading Test dataset...")
X_test, y_test = load_images(test_path, categories)

os.makedirs(r"D:\BioAnnotate_AI\Data\TaxonoML\Skin\Files", exist_ok=True)
np.save(r"D:\BioAnnotate_AI\Data\TaxonoML\Skin\Files\data_train.npy", X_train)
np.save(r"D:\BioAnnotate_AI\Data\TaxonoML\Skin\Files\target_train.npy", y_train)
np.save(r"D:\BioAnnotate_AI\Data\TaxonoML\Skin\Files\data_test.npy", X_test)
np.save(r"D:\BioAnnotate_AI\Data\TaxonoML\Skin\Files\target_test.npy", y_test)

print("Datasets saved successfully!")

Loading Train dataset...


Loading Unknown: 100%|██████████| 19/19 [00:00<00:00, 116.59it/s]


Loading Test dataset...
Datasets saved successfully!


In [5]:
import numpy as np

X_train = np.load(r"D:\BioAnnotate_AI\Data\TaxonoML\Skin\Files\data_train.npy")
y_train = np.load(r"D:\BioAnnotate_AI\Data\TaxonoML\Skin\Files\target_train.npy")

indices = np.arange(X_train.shape[0])
np.random.shuffle(indices)

X_train = X_train[indices]
y_train = y_train[indices]

print("Shuffled Train dataset")
print("X_train shape:", X_train.shape)
print("y_train shape:", y_train.shape)

Shuffled Train dataset
X_train shape: (472, 128, 128, 3)
y_train shape: (472,)


In [1]:
import tensorflow as tf
from tensorflow.keras.utils import to_categorical

print("TensorFlow version:", tf.__version__)

TensorFlow version: 2.21.0


In [3]:
import numpy as np
from tensorflow.keras.utils import to_categorical

X_train = np.load(r"D:\BioAnnotate_AI\Data\TaxonoML\Skin\Files\data_train.npy")
y_train = np.load(r"D:\BioAnnotate_AI\Data\TaxonoML\Skin\Files\target_train.npy")

indices = np.arange(X_train.shape[0])
np.random.shuffle(indices)
X_train = X_train[indices]
y_train = y_train[indices]

y_train = to_categorical(y_train, num_classes=len(np.unique(y_train)))

print("X_train shape:", X_train.shape)
print("y_train shape (one-hot):", y_train.shape)

X_train shape: (472, 128, 128, 3)
y_train shape (one-hot): (472, 10)


In [6]:
print("X_train shape:", X_train.shape)
print("y_train shape:", y_train.shape)

X_train shape: (472, 128, 128, 3)
y_train shape: (472, 10)


In [ ]:
np.save(r"D:\BioAnnotate_AI\Data\TaxonoML\Skin\Files\data_train_processed.npy", X_train)
np.save(r"D:\BioAnnotate_AI\Data\TaxonoML\Skin\Files\target_train_processed.npy", y_train)

In [4]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout
from tensorflow.keras.optimizers import Adam
